In [1]:
from typing import List

import torch
import pytorch_lightning as pl
from torch import Tensor, nn
from torch.nn import functional as F
from torchmetrics import JaccardIndex, F1Score
from torchvision.models import ResNet18_Weights, resnet18
from torchvision.models._utils import IntermediateLayerGetter
from torchvision.models.segmentation.deeplabv3 import ASPP
from pytorch_lightning.loggers import TensorBoardLogger
import torch
import torch.nn as nn
import torch.nn.functional as F
from kornia import losses

from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from pytorch_lightning.callbacks import RichProgressBar
import numpy as np
import glob
import os
import rasterio
import re

In [2]:
def _reshape_maks(inputs, targets, ignore_index: int):
    # do everything using a 1d array
    if inputs.dim() > 2:
        # N,C,H,W => N,C,H*W
        inputs = inputs.view(inputs.size(0), inputs.size(1), -1)
        # N,C,H*W => N,H*W,C
        inputs = inputs.transpose(1, 2)
        # N,H*W,C => N*H*W,C
        inputs = inputs.contiguous().view(-1, inputs.size(2))    

    targets = targets.view(-1, 1)

    # drop ignored_index
    mask = targets==ignore_index
    targets = targets[~mask.ravel(),:].ravel()
    inputs = inputs[~mask.ravel(),:]

    return inputs, targets


class FocalLossMod(nn.Module):
    
    def __init__(self, alpha=0.25, gamma=2.0, reduction='mean', ignore_index=-100):
        super(FocalLossMod, self).__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.reduction = reduction
        self.ignore_index = ignore_index

    def forward(self, inputs_in, targets_in):

        # clone tensors to not disturb tensors that were provided
        inputs = torch.clone(inputs_in)
        targets = torch.clone(targets_in)

        inputs, targets = _reshape_maks(inputs, targets, self.ignore_index)

        return losses.focal_loss(inputs, targets, self.alpha, self.gamma, self.reduction)

class DiceLossMod(nn.Module):
    """Dice Loss modified from kornia to be able to handle "ignore_index"
    https://kornia.readthedocs.io/en/latest/_modules/kornia/losses/dice.html#DiceLoss
    """
    
    def __init__(self, ignore_index=-100):
        super(DiceLossMod, self).__init__()
        self.ignore_index = ignore_index

    def forward(self, inputs_in, targets_in):

        if not isinstance(inputs_in, torch.Tensor):
            raise TypeError(f"Input type is not a torch.Tensor. Got {type(inputs_in)}")

        if not inputs_in.device == targets_in.device:
            raise ValueError(f"input and target must be in the same device. Got: {inputs_in.device} and {targets_in.device}")

        eps: float = 1e-8
        num_classes = inputs_in.shape[1]

        # clone tensors to not disturb tensors that were provided
        inputs = torch.clone(inputs_in)
        targets = torch.clone(targets_in)

        inputs, targets = _reshape_maks(inputs, targets, self.ignore_index)

        # compute softmax over the classes axis
        inputs_soft: torch.Tensor = F.softmax(inputs, dim=1)

        # create the labels one hot tensor
        targets_one_hot: torch.Tensor = F.one_hot(targets, num_classes=num_classes)

        # compute the actual dice score
        intersection = torch.sum(inputs_soft * targets_one_hot)
        cardinality = torch.sum(inputs_soft + targets_one_hot)

        dice_score = 2.0 * intersection / (cardinality + eps)

        return torch.mean(-dice_score + 1.0)


In [ ]:
###############################################
###############################################
class ResNetASPP(pl.LightningModule):
    def __init__(self, *args, **kwargs):
        super().__init__()

        self.save_hyperparameters()

        if kwargs['pretrained'] == True:
            weights=ResNet18_Weights.IMAGENET1K_V1
        else:
            weights=None

        return_layers = {"layer2": "out"}
        self.encoder = resnet18(weights=weights)
        self.encoder = IntermediateLayerGetter(self.encoder, return_layers=return_layers)

        if self.hparams.frozen_start:
            for param in self.encoder.parameters():
                param.requires_grad = False
    
        # create the head:
        self.decoder = nn.Sequential(ASPP(in_channels=128, atrous_rates = [12, 24, 36], out_channels = 128),
                                        nn.Conv2d(128, 128, 3, padding=1, bias=False),
                                        nn.BatchNorm2d(128),
                                        nn.ReLU(),
                                        nn.Dropout2d(p=0.3),
                                        nn.Conv2d(128, self.hparams['num_classes'], 1)
                                        )

        # ASPP has one global average pooling that messes things up 
        # in case we want to change the input size (full raster prediction)
        avgpool_replacer = nn.AvgPool2d(8,8)
        if isinstance(self.decoder[0].convs[-1][0], nn.AdaptiveAvgPool2d):
            self.decoder[0].convs[-1][0] = avgpool_replacer
        else:
            print('Check the model! Is there an AdaptiveAvgPool2d somewhere?')

        # initialize random weights with kaiming normal rather than kaiming uniform
        # for m in self.decoder.modules():
        #     if isinstance(m, nn.Conv2d):
        #         nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
        #     elif isinstance(m, (nn.BatchNorm2d, nn.GroupNorm)):
        #         nn.init.constant_(m.weight, 1)
        #         nn.init.constant_(m.bias, 0)

        # metrics
        # use the num_classes+1 to ignore last index (torchmetrics F1 does not allow arbitrary "ignore_index")
        # torchmetrics Jaccard Index does not work when average='weighted' & ignore_index is present
        # use macro as underperforming classes will have a higher influence in the final value
        self.jaccard = JaccardIndex(num_classes=self.hparams.num_classes+1, 
                                    average='macro',
                                    task="binary")
        self.f1 = F1Score(num_classes=self.hparams.num_classes+1, 
                                    average='macro',
                                    task="binary")

        # loss
        if self.hparams.loss == 'cross_entropy':
            self.loss_fn = nn.CrossEntropyLoss()
        elif self.hparams.loss == 'focal':
            self.loss_fn = FocalLossMod(gamma=self.hparams.gamma, 
                                        alpha=self.hparams.alpha, 
                                        reduction='mean')

        elif self.hparams.loss == 'dice':
            self.loss_fn = DiceLossMod()

        
    def forward(self, x:Tensor) -> Tensor:
        
        input_shape = x.shape[-2:]

        features = self.encoder(x)['out']

        logits = self.decoder(features)         
        logits = F.interpolate(logits, size=input_shape, mode="bilinear", align_corners=False)

        return logits
    
    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr)
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 
                                                                    min_lr=1e-8,
                                                                    patience=self.hparams.reduce_lr_patience, 
                                                                    verbose=True)         
        return optimizer 
    
    def training_step(self, batch, batch_idx):

        x, y = batch
        # torchmetrics F1 does not allow arbitrary "ignore_index"
        y_pred = self.forward(x)
        loss = self.loss_fn(y_pred, y)
        self.log('train_loss', loss, on_epoch=True, prog_bar=True)

        ###############################################
        # metrics
        ###############################################
        # IoU/Jaccard index
        iou = self.jaccard(torch.argmax(y_pred, axis=1), y)
        self.log('train_IoU', iou, on_epoch=True, prog_bar=True)

        # F1 
        f1 = self.f1(torch.argmax(y_pred, axis=1).ravel(), y.ravel())
        self.log('train_f1', f1, on_epoch=True, prog_bar=True)

        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch

        y_pred = self.forward(x)
        loss = self.loss_fn(y_pred, y)
        self.log('val_loss', loss, on_epoch=True, prog_bar=True)

        ###############################################
        # metrics
        ###############################################
        # IoU/Jaccard index
        iou = self.jaccard(torch.argmax(y_pred, axis=1), y)
        self.log('val_IoU', iou, on_epoch=True, prog_bar=True)

        # F1
        f1 = self.f1(torch.argmax(y_pred, axis=1).ravel(), y.ravel())
        self.log('val_f1', f1, on_epoch=True, prog_bar=True)

        return loss

    def on_train_epoch_end(self) -> None:
    # ReduceLROnPlateau expects val_loss
        sch = self.scheduler
        if isinstance(sch, torch.optim.lr_scheduler.ReduceLROnPlateau):
            val_loss = self.trainer.callback_metrics.get("val_loss")
            if val_loss is not None:
                sch.step(val_loss)

                # Unfreeze encoder if frozen_start and learning rate decreased
                if self.hparams.frozen_start and (sch.optimizer.param_groups[0]['lr'] < self.hparams.lr):
                    for param in self.encoder.parameters():
                        param.requires_grad = True
                  

In [4]:
def load_data_new(input_dir, mask_dir):

    # Ensure the directories exist
    if not os.path.exists(input_dir):
        print(f"Input directory {input_dir} does not exist.")
        return None, None
    
    if not os.path.exists(mask_dir):
        print(f"Mask directory {mask_dir} does not exist.")
        return None, None

    # Search for .tif files in the directories
    input_files = glob.glob(os.path.join(input_dir, '*.tif'))
    mask_files = glob.glob(os.path.join(mask_dir, '*.tif'))

    # print("Found input files:", input_files)
    # print("Found mask files:", mask_files)

    if not input_files:
        print(f"No input files found in {input_dir}.")
        return None, None

    if not mask_files:
        print(f"No mask files found in {mask_dir}.")
        return None, None

    images = []
    masks = []

    for mask_file in mask_files:
        # Extract the number i from the mask filename
        match = re.search(r'NDWI_Mask_(\d+)_resized_corrupt.tif', os.path.basename(mask_file))
        if match:
            i = match.group(1)
            input_file = os.path.join(input_dir, f'{i}.tif')
            # print("Input File : ",input_file)
            # Check if the corresponding input file exists
            if os.path.exists(input_file):
                # Read input file
                with rasterio.open(input_file) as src:

                    # print("Width x Height:", src.width, "x", src.height)
                    # print("Number of bands (channels):", src.count)
                    # print("CRS:", src.crs)           # Optional: coordinate reference system
                    # print("Bounds:", src.bounds)     # Optional: extent
                    # print(src.meta)          # General info
                    # print(src.descriptions)
                    img = src.read(1)  # Read the first band assuming it's a single-band image
                    images.append(img)

                # Read mask file
                with rasterio.open(mask_file) as src:
                    
                    # print("Width x Height:", src.width, "x", src.height)
                    # print("Number of bands (channels):", src.count)
                    # print("CRS:", src.crs)           # Optional: coordinate reference system
                    # print("Bounds:", src.bounds) 
                    # print(src.meta)          # General info
                    # print(src.descriptions)
                    msk = src.read(1)  # Read the first band assuming it's a single-band image
                    masks.append(msk)
            else:
                print(f"Corresponding input file {input_file} for mask {mask_file} not found.")

    if not images or not masks:
        print("No matching pairs of images and masks found.")
        return None, None

    return np.array(images), np.array(masks)


def load_data_test(input_dir, mask_dir):

    # Ensure the directories exist
    if not os.path.exists(input_dir):
        print(f"Input directory {input_dir} does not exist.")
        return None, None
    
    if not os.path.exists(mask_dir):
        print(f"Mask directory {mask_dir} does not exist.")
        return None, None

    # Search for .tif files in the directories
    input_files = glob.glob(os.path.join(input_dir, '*.tif'))
    mask_files = glob.glob(os.path.join(mask_dir, '*.tif'))

    # print("Found input files:", input_files)
    # print("Found mask files:", mask_files)

    if not input_files:
        print(f"No input files found in {input_dir}.")
        return None, None

    if not mask_files:
        print(f"No mask files found in {mask_dir}.")
        return None, None

    images = []
    masks = []

    for mask_file in mask_files:
        # Extract the number i from the mask filename
        match = re.search(r'NDWI_Mask_(\d+)_resized.tif', os.path.basename(mask_file))
        if match:
            i = match.group(1)
            input_file = os.path.join(input_dir, f'{i}.tif')
            
            # Check if the corresponding input file exists
            if os.path.exists(input_file):
                # Read input file
                with rasterio.open(input_file) as src:
                    img = src.read(1)  # Read the first band assuming it's a single-band image
                    images.append(img)

                # Read mask file
                with rasterio.open(mask_file) as src:
                    msk = src.read(1)  # Read the first band assuming it's a single-band image
                    masks.append(msk)
            else:
                print(f"Corresponding input file {input_file} for mask {mask_file} not found.")

    if not images or not masks:
        print("No matching pairs of images and masks found.")
        return None, None

    return np.array(images), np.array(masks)



In [5]:
class NDWIDataset(torch.utils.data.Dataset):
    def __init__(self, images=None, masks=None,
                 input_dir=None, mask_dir=None):

        if images is None and masks is None:
            # normal loading
            self.images, self.masks = load_data_new(input_dir, mask_dir)
        else:
            # use already-split arrays
            self.images = images
            self.masks = masks

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx].astype(np.float32)   # (H, W)
        mask = self.masks[idx].astype(np.int64)     # (H, W)

        img = np.repeat(img[None, :, :], 3, axis=0) # (3, H, W)

        img = torch.tensor(img, dtype=torch.float32)
        mask = torch.tensor(mask, dtype=torch.long)

        return img, mask

In [6]:
# Load once
input_dir = './data_new/'
mask_dir = f'./GEE_Masks/GEE_resized/train_gee/train_0_gee_with_diff_kernels'

all_images, all_masks = load_data_new(input_dir=input_dir, mask_dir=mask_dir)

print(all_images.shape)
print(all_masks.shape)

# Split
train_imgs, val_imgs, train_masks, val_masks = train_test_split(
    all_images,
    all_masks,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

c:\Users\ADMIN\anaconda3\envs\aspp_pytorch\lib\site-packages\rasterio\__init__.py:356: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)


(1010, 512, 512)
(1010, 512, 512)


In [8]:
train_dataset = NDWIDataset(images=train_imgs, masks=train_masks)
val_dataset = NDWIDataset(images=val_imgs, masks=val_masks)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=0, pin_memory=True)

model = ResNetASPP(
    pretrained=True,
    frozen_start=False,
    num_classes=2,
    lr=1e-4,
    reduce_lr_patience=3,
    loss="cross_entropy",
    gamma=2.0,
    alpha=0.25
)

logger = TensorBoardLogger("logs", name="resnet_aspp_ndwi")

trainer = pl.Trainer(
    max_epochs=30,
    accelerator="gpu",
    devices=1,
    precision=16,
    log_every_n_steps=20,
    logger=logger,
    # callbacks=[RichProgressBar()]
)

trainer.fit(model, train_loader, val_loader)
trainer.save_checkpoint("resnet_aspp_best.ckpt")

Using 16bit Automatic Mixed Precision (AMP)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name    | Type                    | Params | Mode 
------------------------------------------------------------
0 | encoder | IntermediateLayerGetter | 683 K  | train
1 | decoder | Sequential              | 706 K  | train
2 | jaccard | BinaryJaccardIndex      | 0      | train
3 | f1      | BinaryF1Score           | 0      | train
4 | loss_fn | CrossEntropyLoss        | 0      | train
------------------------------------------------------------
1.4 M     Trainable params
0         Non-trainable params
1.4 M     Total params
5.559     Total estimated model params size (MB)
70        

Epoch 4:  16%|█▌        | 16/101 [00:01<00:06, 12.46it/s, v_num=3, train_loss_step=0.107, train_IoU_step=0.812, train_f1_step=0.896, val_loss=0.155, val_IoU=0.777, val_f1=0.872, train_loss_epoch=0.165, train_IoU_epoch=0.770, train_f1_epoch=0.865] 


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1